In [1]:
import pandas as pd

tickets = pd.read_csv(
    "../data/synthetic/support_tickets.csv"
)

print("Dataset shape:", tickets.shape)

print("\nColumn names:")
print(tickets.columns.tolist())

print("\nFirst 5 records:")
display(tickets.head())

Dataset shape: (600, 9)

Column names:
['ticket_id', 'customer_id', 'order_id', 'ticket_date', 'ticket_text', 'category', 'priority', 'sentiment', 'ticket_status']

First 5 records:


,ticket_id,customer_id,order_id,ticket_date,ticket_text,category,priority,sentiment,ticket_status
0,TKT00001,CUST0306,ORD01232,2026-10-10 04:41:00,Thank you for your support.,Account,Low,Positive,Resolved
1,TKT00002,CUST0211,ORD00869,2026-10-03 11:56:00,Payment failed but money was deducted.,Payment,High,Negative,Resolved
2,TKT00003,CUST0233,ORD00937,2026-07-05 12:24:00,The product quality was excellent.,Product Quality,Low,Positive,Resolved
3,TKT00004,CUST0026,ORD00130,2026-10-19 11:19:00,My payment was deducted twice.,Payment,High,Negative,Resolved
4,TKT00005,CUST0348,ORD01430,2026-06-20 01:28:00,The delivery experience was good.,Delivery,Low,Positive,Resolved


In [2]:
import pandas as pd

tickets = pd.read_csv(
    "../data/synthetic/support_tickets.csv"
)

print("Dataset shape:", tickets.shape)
print("\nColumns:")
print(tickets.columns.tolist())

display(tickets.head())

Dataset shape: (600, 9)

Columns:
['ticket_id', 'customer_id', 'order_id', 'ticket_date', 'ticket_text', 'category', 'priority', 'sentiment', 'ticket_status']


,ticket_id,customer_id,order_id,ticket_date,ticket_text,category,priority,sentiment,ticket_status
0,TKT00001,CUST0306,ORD01232,2026-10-10 04:41:00,Thank you for your support.,Account,Low,Positive,Resolved
1,TKT00002,CUST0211,ORD00869,2026-10-03 11:56:00,Payment failed but money was deducted.,Payment,High,Negative,Resolved
2,TKT00003,CUST0233,ORD00937,2026-07-05 12:24:00,The product quality was excellent.,Product Quality,Low,Positive,Resolved
3,TKT00004,CUST0026,ORD00130,2026-10-19 11:19:00,My payment was deducted twice.,Payment,High,Negative,Resolved
4,TKT00005,CUST0348,ORD01430,2026-06-20 01:28:00,The delivery experience was good.,Delivery,Low,Positive,Resolved


In [3]:
print(tickets["category"].value_counts())

category
Account            132
Payment            127
Delivery           118
Product Quality    115
Refund             108
Name: count, dtype: int64


In [4]:
print(tickets["priority"].value_counts())

priority
High      233
Low       213
Medium    154
Name: count, dtype: int64


In [5]:
print("Ticket Categories:")
print(tickets["category"].value_counts())

print("\nTicket Priorities:")
print(tickets["priority"].value_counts())

print("\nTicket Sentiments:")
print(tickets["sentiment"].value_counts())

print("\nTicket Status:")
print(tickets["ticket_status"].value_counts())

Ticket Categories:
category
Account            132
Payment            127
Delivery           118
Product Quality    115
Refund             108
Name: count, dtype: int64

Ticket Priorities:
priority
High      233
Low       213
Medium    154
Name: count, dtype: int64

Ticket Sentiments:
sentiment
Negative    318
Positive    170
Neutral     112
Name: count, dtype: int64

Ticket Status:
ticket_status
Resolved       343
In Progress    132
Open           125
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

tickets = tickets.dropna(
    subset=["ticket_text", "category"]
).copy()

tickets["ticket_text"] = tickets["ticket_text"].astype(str)

X = tickets["ticket_text"]
y = tickets["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 480
Testing records: 120


In [7]:
ticket_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

ticket_model.fit(X_train, y_train)

print("Ticket classification model trained!")

Ticket classification model trained!


In [8]:
y_pred = ticket_model.predict(X_test)

print(
    "Accuracy:",
    round(accuracy_score(y_test, y_pred) * 100, 2),
    "%"
)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 100.0 %

Classification Report:
                 precision    recall  f1-score   support

        Account       1.00      1.00      1.00        26
       Delivery       1.00      1.00      1.00        24
        Payment       1.00      1.00      1.00        25
Product Quality       1.00      1.00      1.00        23
         Refund       1.00      1.00      1.00        22

       accuracy                           1.00       120
      macro avg       1.00      1.00      1.00       120
   weighted avg       1.00      1.00      1.00       120



In [9]:
new_ticket = [
    "My order was delayed and I have not received my refund"
]

predicted_category = ticket_model.predict(new_ticket)

print("Predicted Category:", predicted_category[0])

Predicted Category: Delivery


In [10]:
import joblib

joblib.dump(
    ticket_model,
    "../models/ticket_classifier.pkl"
)

print("Ticket classification model saved!")

Ticket classification model saved!
